# Deploying DeepSeek-V4-Flash with SGLang: Research Paper Synthesis

This notebook will walk you through how to run the [`deepseek-ai/DeepSeek-V4-Flash`](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash) model on **2× NVIDIA B200s** using [SGLang](https://github.com/sgl-project/sglang), and use its 1M-token context window to synthesize insights across a live bundle of research papers pulled from arXiv.

[SGLang](https://github.com/sgl-project/sglang) is a fast serving framework for large language models with RadixAttention for efficient KV cache reuse.

For more details on the model, [click here](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash).

> **Note on GPU count:** SGLang's validated recipe for DeepSeek-V4-Flash on B200 uses 4 GPUs (`--tp 4`). This notebook runs with 2 GPUs (`--tp 2`) as a starting point — it will work but is not on the fully benchmarked path. To match the validated configuration, change `--tp 2` to `--tp 4` and update `"device=0,1"` to `"device=0,1,2,3"` accordingly. See the [SGLang DeepSeek-V4 cookbook](https://lmsysorg.mintlify.app/cookbook/autoregressive/DeepSeek/DeepSeek-V4) for all hardware recipes.

Prerequisites:
- NVIDIA GPU(s) with recent drivers (this notebook uses 2×B200)
- A Hugging Face account with access to [`deepseek-ai/DeepSeek-V4-Flash`](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash)
- Internet access for the arXiv API


## Overview

- **Serve** DeepSeek-V4-Flash on 2× NVIDIA B200s using SGLang
- **Fetch** a live bundle of research papers from the arXiv API — no account required
- **Synthesize** a research map, theme clusters, open problems, and cross-paper comparisons
- **Compare** Non-Think and Think-High reasoning modes on a complex synthesis question

## Table of Contents

1. **Install dependencies** — client packages
2. **Start the SGLang server** — Docker command with configuration reference
3. **Fetch research papers** — arXiv API search, optional PDF download
4. **Build the research bundle** — context assembly
5. **Research map** — field overview, themes, open problems
6. **Deep-dive questions** — cross-paper methodology and benchmarking analysis
7. **Non-Think vs Think-High** — reasoning mode comparison

## Step 0 — Install Dependencies

In [ ]:
%pip install --quiet --upgrade openai arxiv pypdf requests

In [ ]:
import os

# Paste your token here if you haven't run `huggingface-cli login`
# os.environ["HF_TOKEN"] = "hf_..."

## Step 1 — Start the SGLang Server

Run the following command in a separate terminal before continuing. Leave the server running while this notebook executes.

### Docker (recommended)

The `lmsysorg/sglang:latest` image includes the full CUDA toolkit, so FP4/FP8 JIT kernels compile correctly without requiring any additional setup on the host.

```bash
docker run --gpus '"device=0,1"' \
    --shm-size 32g \
    -p 30000:30000 \
    -v ~/.cache/huggingface:/root/.cache/huggingface \
    --ipc=host \
    lmsysorg/sglang:latest \
    sglang serve \
      --trust-remote-code \
      --model-path deepseek-ai/DeepSeek-V4-Flash \
      --tp 2 \
      --moe-runner-backend flashinfer_mxfp4 \
      --speculative-algorithm EAGLE \
      --speculative-num-steps 3 \
      --speculative-eagle-topk 1 \
      --speculative-num-draft-tokens 4 \
      --chunked-prefill-size 4096 \
      --disable-flashinfer-autotune \
      --swa-full-tokens-ratio 0.1 \
      --reasoning-parser deepseek-v4 \
      --host 0.0.0.0 \
      --port 30000
```

> **Note:** `~/.cache/huggingface` is mounted into the container so that previously downloaded model weights are reused automatically. To use 4 GPUs (the fully validated recipe), change `"device=0,1"` to `"device=0,1,2,3"` and `--tp 2` to `--tp 4`. See the [SGLang DeepSeek-V4 cookbook](https://lmsysorg.mintlify.app/cookbook/autoregressive/DeepSeek/DeepSeek-V4) for all hardware and recipe combinations.

### Configuration reference

| Flag | Purpose |
|---|---|
| `--tp 2` | Tensor-parallel across 2 GPUs (use `--tp 4` for the validated recipe) |
| `--moe-runner-backend flashinfer_mxfp4` | FP4 MoE kernel required for DeepSeek-V4's expert weights |
| `--speculative-algorithm EAGLE` | EAGLE speculative decoding; reduces time-to-first-token at low batch sizes |
| `--chunked-prefill-size 4096` | Chunked prefill for stable long-context processing |
| `--reasoning-parser deepseek-v4` | Parses `<think>...</think>` blocks and exposes `reasoning_content` in responses |

> **Expected output:** Wait for `The server is fired up and ready to roll!` before continuing. On first run, model weights will be downloaded from Hugging Face; this may take several minutes.

In [1]:
import requests, time

SGLANG_BASE_URL = "http://localhost:30000"

for attempt in range(30):
    try:
        if requests.get(f"{SGLANG_BASE_URL}/health", timeout=5).status_code == 200:
            print("Server ready.")
            break
    except requests.exceptions.ConnectionError:
        pass
    print(f"Waiting... ({attempt + 1}/30)")
    time.sleep(10)

Server ready.


In [2]:
from openai import OpenAI

client = OpenAI(base_url=f"{SGLANG_BASE_URL}/v1", api_key="not-needed")
MODEL  = "deepseek-ai/DeepSeek-V4-Flash"

print([m.id for m in client.models.list().data])

['deepseek-ai/DeepSeek-V4-Flash']


## Step 2 — Fetch Research Papers from arXiv

The [`arxiv`](https://github.com/lukasschwab/arxiv.py) client retrieves papers through the [arXiv API](https://arxiv.org/help/api/) — no account required.

The default query targets LLM inference papers, which are directly relevant to the frameworks and model architecture covered in this notebook. Set `SEARCH_QUERY` to any topic of interest. Full arXiv query syntax is documented at [arxiv.org/help/api/user-manual](https://arxiv.org/help/api/user-manual#query_details).

In [4]:
import arxiv
import time

SEARCH_QUERY = (
    'ti:"LLM inference" OR ti:"efficient inference" OR ti:"speculative decoding" '
    'OR ti:"continuous batching" OR ti:"KV cache" OR ti:"paged attention"'
)
MAX_PAPERS = 20
MAX_ABSTRACT_CHARS = 2000

search = arxiv.Search(
    query=SEARCH_QUERY,
    max_results=MAX_PAPERS,
    sort_by=arxiv.SortCriterion.SubmittedDate,
    sort_order=arxiv.SortOrder.Descending,
)

papers = []
for result in arxiv.Client(delay_seconds=3, num_retries=5).results(search):
    papers.append({
        "id"        : result.entry_id.split("/")[-1],
        "title"     : result.title,
        "authors"   : ", ".join(a.name for a in result.authors[:5])
                       + (" et al." if len(result.authors) > 5 else ""),
        "published" : result.published.strftime("%Y-%m-%d"),
        "abstract"  : result.summary.replace("\n", " ")[:MAX_ABSTRACT_CHARS],
        "categories": ", ".join(result.categories),
        "pdf_url"   : result.pdf_url,
    })
    time.sleep(1)  # extra courtesy delay between results

for i, p in enumerate(papers, 1):
    print(f"{i:>2}. [{p['published']}] {p['title']}")
    print(f"     {p['authors']}")

 1. [2026-05-29] GRKV: Global Regression for Training-Free KV Cache Compression in Long-Context LLMs
     Junjie Peng, You Wu, Haoyi Wu, Jialong Han, Xiaohua Xie et al.
 2. [2026-05-28] Speculative Decoding Across Languages
     Nirajan Paudel, Michael Ginn, Luc De Nardi, Alexis Palmer
 3. [2026-05-28] Probing the Prompt KV Cache: Where It Becomes Dispensable
     Vinayshekhar Bannihatti Kumar, Manoj Ghuhan Arivazhagan, Disha Makhija, Rashmi Gangadharaiah
 4. [2026-05-28] VideoMLA: Low-Rank Latent KV Cache for Minute-Scale Autoregressive Video Diffusion
     Hidir Yesiltepe, Jiazhen Hu, Tuna Han Salih Meral, Adil Kaan Akan, Kaan Oktay et al.
 5. [2026-05-28] MarginGate: Sparse Margin-Triggered Verification for Batch-Invariant LLM Inference
     Kexin Chu, Yang Zhou, Wei Zhang
 6. [2026-05-28] Future Forcing: Future-aware Training-free KV Cache Policy for Autoregressive Video Generation
     Jiayi Luo, Qiyan Liu, Tengyang Wang, JunHao Liu, Jiayu Chen et al.
 7. [2026-05-28] Moment-KV: M

### Optional — Download Full Paper Text

Downloads PDFs for the first `N_FULL_PAPERS` results and extracts their text. Skip this step if abstracts are sufficient for your use case.

In [5]:
import pathlib
import pypdf

N_FULL_PAPERS = 5
MAX_PDF_CHARS = 30_000
PDF_CACHE_DIR = pathlib.Path("./arxiv_pdfs")
PDF_CACHE_DIR.mkdir(exist_ok=True)


def download_and_extract(paper: dict) -> str:
    cache_path = PDF_CACHE_DIR / f"{paper['id']}.pdf"
    if not cache_path.exists():
        resp = requests.get(paper["pdf_url"], timeout=60)
        resp.raise_for_status()
        cache_path.write_bytes(resp.content)
    text = ""
    for page in pypdf.PdfReader(str(cache_path)).pages:
        text += page.extract_text() or ""
        if len(text) >= MAX_PDF_CHARS:
            break
    return text[:MAX_PDF_CHARS]


for i, paper in enumerate(papers[:N_FULL_PAPERS]):
    try:
        paper["full_text"] = download_and_extract(paper)
        print(f"[{i+1}/{N_FULL_PAPERS}] {paper['title'][:70]} — {len(paper['full_text']):,} chars")
    except Exception as e:
        paper["full_text"] = None
        print(f"[{i+1}/{N_FULL_PAPERS}] {paper['title'][:70]} — failed: {e}")

[1/5] GRKV: Global Regression for Training-Free KV Cache Compression in Long — 30,000 chars
[2/5] Speculative Decoding Across Languages — 30,000 chars
[3/5] Probing the Prompt KV Cache: Where It Becomes Dispensable — 30,000 chars
[4/5] VideoMLA: Low-Rank Latent KV Cache for Minute-Scale Autoregressive Vid — 30,000 chars
[5/5] MarginGate: Sparse Margin-Triggered Verification for Batch-Invariant L — 30,000 chars


---
## Step 3 — Build the Research Bundle

In [6]:
def format_paper(paper: dict, index: int) -> str:
    parts = [
        f"## Paper {index}: {paper['title']}",
        f"**Authors:** {paper['authors']}",
        f"**Published:** {paper['published']}  |  **arXiv ID:** {paper['id']}",
        f"**Categories:** {paper['categories']}",
        "",
        "**Abstract:**",
        paper["abstract"],
    ]
    if paper.get("full_text"):
        parts += ["", "**Full Text (excerpt):**", paper["full_text"]]
    return "\n".join(parts)


bundle_sections = [f"# Research Bundle: Efficient LLM Inference\n**Papers:** {len(papers)}"]
for i, paper in enumerate(papers, 1):
    bundle_sections.append(format_paper(paper, i))

research_bundle = "\n\n---\n\n".join(bundle_sections)

print(f"{len(research_bundle):,} chars  |  ~{len(research_bundle) // 4:,} tokens  |  "
      f"{sum(1 for p in papers if p.get('full_text'))} papers with full text")

182,394 chars  |  ~45,598 tokens  |  5 papers with full text


## Step 4 — Generate a Research Map

This is the initial call and the longest — it encodes the full research bundle into the KV cache. Subsequent queries reuse that cache automatically via SGLang's RadixAttention.

In [7]:
RESEARCH_MAP_PROMPT = """\
Here is a bundle of research papers on efficient LLM inference:

{bundle}

---

Produce a Research Map with these sections:

1. **Field Overview** (3–5 sentences): The core problem and why it matters now.

2. **Research Themes**: 4–6 distinct themes across these papers. For each:
   - Name and one-sentence description
   - Which papers belong to it (by number and title)
   - Key insight that defines the theme

3. **Chronological Progression**: How has the field evolved? What ideas were superseded?

4. **Open Problems**: 3–5 important unsolved problems.

5. **Practitioner Takeaways**: Which 3 ideas are most ready to deploy today?
"""

response = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": RESEARCH_MAP_PROMPT.format(bundle=research_bundle)}],
    max_tokens=3000,
    temperature=0.3,
)
print(response.choices[0].message.content)

Here is a Research Map synthesizing the key findings from the provided bundle of 20 papers on efficient LLM inference.

### 1. Field Overview

The core problem across this research bundle is the escalating memory and computational bottleneck of the **Key-Value (KV) cache** during Large Language Model (LLM) inference, particularly as context lengths grow and models are deployed at scale. This bottleneck is critical now because the shift toward long-context, multi-turn, and multi-modal applications (like video generation and reasoning) has made the cache's memory footprint the primary constraint on throughput, latency, and batch size, often outweighing the cost of the model weights themselves. The field is actively moving beyond simple eviction to explore sophisticated *merging*, *quantization*, *speculative*, and *architectural* strategies that treat the cache not as a static store but as a compressible, dynamic, and even partially redundant resource.

### 2. Research Themes

**1. Train

## Step 5 — Deep-Dive Cross-Paper Questions

These queries run against the same context as Step 4. SGLang's RadixAttention reuses the cached KV prefix from the previous call, so each subsequent query is faster than the first.

Edit `DEEP_DIVE_QUESTIONS` to explore the topics most relevant to your work.

In [8]:
DEEP_DIVE_QUESTIONS = [
    (
        "Methodology comparison",
        "Compare the memory management strategies used across the papers. "
        "Which papers share the same fundamental approach, and which ones represent genuine departures? "
        "Cite specific paper numbers."
    ),
    (
        "Benchmarking gaps",
        "Which evaluation benchmarks and metrics appear most often across these papers? "
        "What important real-world performance dimensions are missing from the evaluations?"
    ),
    (
        "Implementation guidance",
        "A small team wants to build a production LLM inference system from scratch. "
        "Synthesize the most actionable engineering lessons from this bundle into a prioritized roadmap."
    ),
]

for label, question in DEEP_DIVE_QUESTIONS:
    print(f"\n[{label}]")
    print("─" * 60)
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": "Reference specific papers by number and title. Be analytical and concrete."},
            {"role": "user",   "content": f"Research bundle:\n\n{research_bundle}\n\n---\n\nQuestion: {question}"},
        ],
        max_tokens=1200,
        temperature=0.2,
    )
    print(resp.choices[0].message.content)
    print(f"\n[prompt: {resp.usage.prompt_tokens:,}  completion: {resp.usage.completion_tokens:,} tokens]")


[Methodology comparison]
────────────────────────────────────────────────────────────
# Memory Management Strategies Across the Papers: A Comparative Analysis

## Overview of Fundamental Approaches

The papers in this research bundle address memory management in LLM inference through several distinct strategies. I'll categorize them by their core mechanisms and identify which share fundamental approaches versus which represent genuine departures.

## Category 1: KV Cache Compression via Eviction and Merging

**Papers sharing the same fundamental approach:** Papers 1, 6, 7, 17

These papers all operate on the principle that the KV cache contains redundant or selectively important information that can be compressed by retaining only a subset of entries. They share the following core characteristics:

- **Paper 1 (GRKV)** and **Paper 6 (Future Forcing)** both use **training-free** methods that compress by identifying which tokens to retain. GRKV uses ridge regression to minimize attentio

## Step 6 — Non-Think vs Think-High

Complex cross-paper tradeoff questions are where Think-High mode is most useful. This step poses a question that requires weighing conflicting evidence across multiple papers.

| Mode | Triggered by | When to use |
|---|---|---|
| Non-Think | Default (no `extra_body`) | Summaries, factual retrieval |
| Think-High | `chat_template_kwargs={"thinking": True}` in `extra_body` | Conflicting evidence, hard tradeoffs |
| Think-Max | `chat_template_kwargs={"thinking": True, "reasoning_effort": "max"}` in `extra_body` ([see model card](https://huggingface.co/deepseek-ai/DeepSeek-V4-Flash)) | Frontier synthesis |

> **Note:** Reasoning content is returned in `choices[0].message.reasoning_content`; if that field is absent, check `choices[0].message.model_extra` (field name varies by SGLang version).

In [9]:
HARD_QUESTION = (
    "Speculative decoding and continuous batching are both presented as solutions to LLM inference latency. "
    "Based on the papers in this bundle, under what conditions does each approach win? "
    "Are they complementary or in tension? "
    "If you had to pick one for a production deployment serving mixed workloads (short and long outputs, "
    "bursty traffic), which would you recommend and why?"
)

base_message = [
    {"role": "user", "content": f"Research bundle:\n\n{research_bundle}\n\n---\n\nQuestion: {HARD_QUESTION}"}
]

resp_nothink = client.chat.completions.create(
    model=MODEL,
    messages=base_message,
    max_tokens=1500,
    temperature=0.2,
)

resp_think = client.chat.completions.create(
    model=MODEL,
    messages=base_message,
    max_tokens=4000,
    temperature=0.6,
    extra_body={"chat_template_kwargs": {"thinking": True}},
)

In [ ]:
print("NON-THINK")
print("─" * 60)
print(resp_nothink.choices[0].message.content)

think_msg = resp_think.choices[0].message
reasoning = (
    getattr(think_msg, "reasoning_content", None)
    or (think_msg.model_extra or {}).get("reasoning_content")
    or (think_msg.model_extra or {}).get("reasoning")
)

print("\n\nTHINK-HIGH")
print("─" * 60)
if reasoning:
    print(f"[reasoning — {len(reasoning.split()):,} words]\n")
    print(reasoning)
    print("\n[answer]\n")
print(think_msg.content)

reasoning_words = len(reasoning.split()) if reasoning else 0
answer_words    = len((think_msg.content or "").split())

print(f"\nTOKEN SUMMARY")
print(f"  Non-think  completion tokens : {resp_nothink.usage.completion_tokens:>6,}")
print(f"  Think-High completion tokens : {resp_think.usage.completion_tokens:>6,}")
print(f"    ├─ reasoning block (~words): {reasoning_words:>6,}")
print(f"    └─ final answer  (~words)  : {answer_words:>6,}")
print(f"  Extra tokens spent thinking  : {resp_think.usage.completion_tokens - resp_nothink.usage.completion_tokens:>+6,}")